# Imputation Experiments: 2-Arm Problem

Isolated 2-arm toy problem for experimenting with imputation strategies.
The key finding: SPO+ under partial feedback fails because the solver gets
wrong cost comparisons from imputed values, not because the gradient is masked.

In [ ]:
import numpy as np
import pandas as pd
from lets_plot import *
from lets_plot.mapping import as_discrete
from scipy.stats import spearmanr
import warnings

LetsPlot.setup_html()
import os
os.makedirs('figures', exist_ok=True)

np.random.seed(1)
LR = 0.03

def solve_topk_min(costs, k):
    sol = np.zeros_like(costs)
    idx = np.argsort(costs)[:k]
    sol[idx] = 1.0
    return sol

def compute_flip_set(pred, true_costs, k):
    solver = lambda c: solve_topk_min(c, k)
    w_true = solver(true_costs)
    w_spo = solver(2 * pred - true_costs)
    flip_mask = np.abs(w_spo - w_true) > 0.5
    return flip_mask, flip_mask.sum()

class BayesianSharedFeatureAgent:
    
    def __init__(self, input_dim, hidden_dim, n_arms, k, lr=None,
                 loss_type='mse', explore='greedy', beta=1.0,
                 lambda_reg=1.0, mask_grad=True, seed=0, decay_rounds=300, beta_end=None,
                 spo_weight=0.5, lr_schedule='constant',
                 phased=False, phase_length=50, phase_epochs=5,
                 mask_full=False, impute_mode='model', parallel_mse=False,
                 bayesian_head=True):
        lr = lr or LR
        self.n_arms = n_arms
        self.k = k
        self.lr = lr
        self.lr_schedule = lr_schedule
        self.loss_type = loss_type
        self.spo_weight = spo_weight
        self.explore = explore
        self.beta = beta
        self.mask_grad = mask_grad
        self.mask_full = mask_full
        self.impute_mode = impute_mode
        self._cost_sum = np.zeros(n_arms)
        self._cost_count = np.zeros(n_arms)
        self._err_sum_sq = np.zeros(n_arms)  # running sum of squared errors
        self.beta_start = beta
        self.beta_end = beta_end if beta_end is not None else beta
        self.decay_rounds = decay_rounds
        self.t = 0
        self.solver = lambda c: solve_topk_min(c, k)
        self.hidden_dim = hidden_dim
        
        rng = np.random.RandomState(seed)
        self.A = rng.randn(hidden_dim, input_dim) * 0.1
        self.bias = np.zeros(hidden_dim)
        
        self.bayesian_head = bayesian_head
        self.Lambda = np.array([lambda_reg * np.eye(hidden_dim) for _ in range(n_arms)])
        self.Lambda_inv = np.array([np.eye(hidden_dim) / lambda_reg for _ in range(n_arms)])
        self.b_vec = np.zeros((n_arms, hidden_dim))
        # Point-estimate head (used when bayesian_head=False)
        self.W_pred = np.zeros((n_arms, hidden_dim))
        self.lambda_reg = lambda_reg
        self.input_dim = input_dim
        self.phased = phased
        self.phase_length = phase_length
        self.phase_epochs = phase_epochs
        if phased:
            self._buf_x, self._buf_y, self._buf_sel = [], [], []
            self._rounds_since_retrain = 0
        
        # Parallel MSE imputer: per-arm OLS on raw (augmented) context.
        # No shared features, no Bayesian head — just w_i^T [x, 1] via closed-form ridge regression.
        self.parallel_mse = parallel_mse
        if parallel_mse:
            self._aug_dim = input_dim + 1
            self.M_mse_inv = np.array([np.eye(self._aug_dim) / lambda_reg for _ in range(n_arms)])
            self.b_mse_raw = np.zeros((n_arms, self._aug_dim))
    
    def _aug(self, x):
        return np.concatenate([x, [1.0]])
    
    def _predict_mse(self, x):
        x_aug = self._aug(x)
        means = np.zeros(self.n_arms)
        for i in range(self.n_arms):
            w_i = self.M_mse_inv[i] @ self.b_mse_raw[i]
            means[i] = w_i @ x_aug
        return means
    
    def _features(self, x):
        return self.A @ x + self.bias
    
    def _head_weight(self, i):
        """Return the prediction weight vector for arm i (chain-rule multiplier)."""
        if self.bayesian_head:
            return self.Lambda_inv[i] @ self.b_vec[i]
        return self.W_pred[i]

    def _predict(self, h):
        means = np.zeros(self.n_arms)
        for i in range(self.n_arms):
            means[i] = self._head_weight(i) @ h
        return means
    
    def _uncertainty(self, h):
        if not self.bayesian_head:
            # No posterior uncertainty available — return zeros
            return np.zeros(self.n_arms)
        stds = np.zeros(self.n_arms)
        for i in range(self.n_arms):
            var = h @ self.Lambda_inv[i] @ h
            stds[i] = np.sqrt(max(var, 0.0))
        return stds
    
    def act(self, x):
        self.t += 1
        if self.beta_start != self.beta_end:
            progress = min(self.t / self.decay_rounds, 1.0)
            self.beta = self.beta_start + (self.beta_end - self.beta_start) * progress
        h = self._features(x)
        means = self._predict(h)
        stds = self._uncertainty(h)
        
        if self.explore == 'greedy':
            return self.solver(means)
        elif self.explore == 'ucb':
            values = means - self.beta * stds
            return self.solver(values)
        elif self.explore == 'emp_ucb':
            # UCB1-style: bonus based on observation count (context-free)
            bonus = np.sqrt(np.log(max(self.t, 2)) / np.maximum(self._cost_count, 1))
            values = means - self.beta * bonus
            return self.solver(values)
        return self.solver(means)
    
    def _effective_lr(self):
        if self.lr_schedule == 'inv_sqrt':
            return self.lr / np.sqrt(max(self.t, 1))
        return self.lr
    
    def _compute_grad(self, x, true_costs, obs_mask, full_feedback):
        """Compute feature gradient for one data point."""
        h = self._features(x)
        pred = self._predict(h)
        if self.loss_type in ('mse', 'hybrid_phased', 'adaptive_phased'):
            grad_pred = np.zeros(self.n_arms)
            for i in range(self.n_arms):
                if obs_mask[i]:
                    grad_pred[i] = 2 * (pred[i] - true_costs[i]) / self.n_arms
        elif self.loss_type == 'spo':
            if full_feedback:
                imputed = true_costs.copy()
                w_imp = self.solver(imputed)
                w_spo = self.solver(2 * pred - imputed)
                grad_pred = 2 * (w_imp - w_spo) / self.n_arms
            elif self.impute_mode.startswith('posterior'):
                # Posterior predictive imputation: c_tilde ~ N(pred, sigma^2)
                # Unbiased MC estimate of E_{c|data}[grad L(pred, c)]
                parts = self.impute_mode.split('_')
                M = int(parts[1]) if len(parts) > 1 else 1
                stds = self._uncertainty(h)
                grad_pred = np.zeros(self.n_arms)
                for _ in range(M):
                    imputed = pred + np.random.randn(self.n_arms) * stds
                    imputed[obs_mask] = true_costs[obs_mask]
                    w_imp = self.solver(imputed)
                    w_spo = self.solver(2 * pred - imputed)
                    grad_pred += 2 * (w_imp - w_spo) / self.n_arms
                grad_pred /= M
            elif self.impute_mode.startswith('empirical'):
                # Use empirical prediction error as noise scale
                parts = self.impute_mode.split('_')
                M = int(parts[1]) if len(parts) > 1 else 1
                emp_var = self._err_sum_sq / np.maximum(self._cost_count, 1)
                emp_std = np.sqrt(np.maximum(emp_var, 1e-4))
                grad_pred = np.zeros(self.n_arms)
                for _ in range(M):
                    imputed = pred + np.random.randn(self.n_arms) * emp_std
                    imputed[obs_mask] = true_costs[obs_mask]
                    w_imp = self.solver(imputed)
                    w_spo = self.solver(2 * pred - imputed)
                    grad_pred += 2 * (w_imp - w_spo) / self.n_arms
                grad_pred /= M
            elif self.impute_mode == 'mean':
                imputed = np.where(self._cost_count > 0,
                                   self._cost_sum / np.maximum(self._cost_count, 1),
                                   0.5 * np.ones(self.n_arms))
                imputed[obs_mask] = true_costs[obs_mask]
                w_imp = self.solver(imputed)
                w_spo = self.solver(2 * pred - imputed)
                grad_pred = 2 * (w_imp - w_spo) / self.n_arms
            elif self.impute_mode == 'mse_parallel':
                # Use parallel MSE model's predictions for imputation
                mse_pred = self._predict_mse(x)
                imputed = mse_pred.copy()
                imputed[obs_mask] = true_costs[obs_mask]
                w_imp = self.solver(imputed)
                w_spo = self.solver(2 * pred - imputed)
                grad_pred = 2 * (w_imp - w_spo) / self.n_arms
            else:
                imputed = pred.copy()
                imputed[obs_mask] = true_costs[obs_mask]
                w_imp = self.solver(imputed)
                w_spo = self.solver(2 * pred - imputed)
                grad_pred = 2 * (w_imp - w_spo) / self.n_arms
            if self.mask_grad and (not full_feedback or self.mask_full):
                grad_pred = np.where(obs_mask, grad_pred, 0.0)
        grad_h = np.zeros(self.hidden_dim)
        for i in range(self.n_arms):
            grad_h += grad_pred[i] * self._head_weight(i)
        return grad_h

    def _update_bayesian(self, h, true_costs, obs_mask):
        """Update prediction head for observed arms.
        If bayesian_head: closed-form ridge regression via Sherman-Morrison.
        If not: SGD on MSE for the point-estimate head W_pred.
        """
        self._cost_sum[obs_mask] += true_costs[obs_mask]
        self._cost_count[obs_mask] += 1
        # Track prediction errors for empirical calibration
        pred_here = self._predict(h)
        self._err_sum_sq[obs_mask] += (pred_here[obs_mask] - true_costs[obs_mask])**2
        if self.bayesian_head:
            for i in range(self.n_arms):
                if obs_mask[i]:
                    self.Lambda[i] += np.outer(h, h)
                    self.b_vec[i] += h * true_costs[i]
                    Lih = self.Lambda_inv[i] @ h
                    self.Lambda_inv[i] -= np.outer(Lih, Lih) / (1.0 + h @ Lih)
        else:
            # SGD on MSE for the point-estimate head
            eff_lr = self._effective_lr()
            for i in range(self.n_arms):
                if obs_mask[i]:
                    self.W_pred[i] -= eff_lr * 2.0 * (pred_here[i] - true_costs[i]) * h

    def _reset_bayesian(self):
        """Reset head stats (for replay after phased retrain)."""
        self.Lambda = np.array([self.lambda_reg * np.eye(self.hidden_dim) for _ in range(self.n_arms)])
        self.Lambda_inv = np.array([np.eye(self.hidden_dim) / self.lambda_reg for _ in range(self.n_arms)])
        self.b_vec = np.zeros((self.n_arms, self.hidden_dim))
        self.W_pred = np.zeros((self.n_arms, self.hidden_dim))

    def _compute_spo_grad_offline(self, x, true_costs, obs_mask):
        """SPO+ gradient using retrospective imputation from current posterior."""
        h = self._features(x)
        pred = self._predict(h)
        # Retrospective imputation: use current posterior (accumulated over full phase)
        imputed = pred.copy()
        imputed[obs_mask] = true_costs[obs_mask]
        w_imp = self.solver(imputed)
        w_spo = self.solver(2 * pred - imputed)
        grad_pred = 2 * (w_imp - w_spo) / self.n_arms
        grad_pred = np.where(obs_mask, grad_pred, 0.0)
        grad_h = np.zeros(self.hidden_dim)
        for i in range(self.n_arms):
            grad_h += grad_pred[i] * self._head_weight(i)
        return grad_h

    def _phased_retrain(self):
        """Batch retrain features, then replay buffer to rebuild posteriors."""
        # Choose gradient based on loss type:
        # - hybrid_phased: SPO+ offline (with online MSE updates from update())
        # - adaptive_phased: per-round mix of SPO+ and MSE based on observation rate
        # - others: use their own loss
        use_spo_offline = (self.loss_type == 'hybrid_phased')
        adaptive = (self.loss_type == 'adaptive_phased')
        # Threshold: only use SPO+ on rounds where >= 50% of arms were observed
        # AND posterior on observed arms is reasonably tight
        spo_threshold = 0.5
        for _epoch in range(self.phase_epochs):
            for x, y, sel in zip(self._buf_x, self._buf_y, self._buf_sel):
                obs_mask = sel > 0.5
                if adaptive:
                    obs_rate = obs_mask.sum() / self.n_arms
                    if obs_rate >= spo_threshold:
                        grad_h = self._compute_spo_grad_offline(x, y, obs_mask)
                    else:
                        # Use MSE on this round
                        h_local = self._features(x)
                        pred = self._predict(h_local)
                        grad_pred = np.zeros(self.n_arms)
                        for i in range(self.n_arms):
                            if obs_mask[i]:
                                grad_pred[i] = 2 * (pred[i] - y[i]) / self.n_arms
                        grad_h = np.zeros(self.hidden_dim)
                        for i in range(self.n_arms):
                            mu_i = self.Lambda_inv[i] @ self.b_vec[i]
                            grad_h += grad_pred[i] * mu_i
                elif use_spo_offline:
                    grad_h = self._compute_spo_grad_offline(x, y, obs_mask)
                else:
                    grad_h = self._compute_grad(x, y, obs_mask, False)
                eff_lr = self._effective_lr()
                self.A -= eff_lr * np.outer(grad_h, x)
                self.bias -= eff_lr * grad_h
        # Replay to rebuild posteriors with updated features
        self._reset_bayesian()
        for x, y, sel in zip(self._buf_x, self._buf_y, self._buf_sel):
            h = self._features(x)
            obs_mask = sel > 0.5
            self._update_bayesian(h, y, obs_mask)

    def _update_parallel_mse(self, x, true_costs, obs_mask):
        """Train the parallel MSE imputer: per-arm OLS on raw (augmented) context.
        Closed-form ridge regression via Sherman-Morrison, no gradient, no solver.
        """
        x_aug = self._aug(x)
        for i in range(self.n_arms):
            if obs_mask[i]:
                Mx = self.M_mse_inv[i] @ x_aug
                self.M_mse_inv[i] -= np.outer(Mx, Mx) / (1.0 + x_aug @ Mx)
                self.b_mse_raw[i] += x_aug * true_costs[i]

    def update(self, x, selection, true_costs, full_feedback=False):
        h = self._features(x)
        if full_feedback:
            obs_mask = np.ones(self.n_arms, dtype=bool)
        else:
            obs_mask = selection > 0.5
        
        self._update_bayesian(h, true_costs, obs_mask)
        
        # Train parallel MSE model every round (safe, no solver contamination)
        if self.parallel_mse:
            self._update_parallel_mse(x, true_costs, obs_mask)

        if self.phased:
            self._buf_x.append(x.copy())
            self._buf_y.append(true_costs.copy())
            self._buf_sel.append(selection.copy())
            
            # Hybrid: apply online MSE gradient every round (safe, no solver contamination)
            if self.loss_type == 'hybrid_phased':
                grad_h = self._compute_grad(x, true_costs, obs_mask, full_feedback)
                eff_lr = self._effective_lr()
                self.A -= eff_lr * np.outer(grad_h, x)
                self.bias -= eff_lr * grad_h
            
            self._rounds_since_retrain += 1
            if self._rounds_since_retrain >= self.phase_length:
                self._phased_retrain()
                self._rounds_since_retrain = 0
        else:
            grad_h = self._compute_grad(x, true_costs, obs_mask, full_feedback)
            eff_lr = self._effective_lr()
            self.A -= eff_lr * np.outer(grad_h, x)
            self.bias -= eff_lr * grad_h



def run_experiment(W_true, k, n_rounds=3000, hidden_dim=8,
                   lr=None, loss_type='mse', explore='greedy',
                   beta=1.0, mask_grad=True, full_feedback=False, seed=0,
                   context_dist='gaussian', ignore_context=False,
                   spo_weight=0.5, lr_schedule='constant',
                 phased=False, phase_length=50, phase_epochs=5,
                   mask_full=False, impute_mode='model', parallel_mse=False,
                   bayesian_head=True, cost_fn='sigmoid'):
    lr = lr or LR
    rng = np.random.RandomState(seed)
    n_arms, input_dim = W_true.shape

    agent = BayesianSharedFeatureAgent(
        input_dim, hidden_dim, n_arms, k, lr=lr,
        loss_type=loss_type, explore=explore, beta=beta,
        mask_grad=mask_grad, seed=seed, spo_weight=spo_weight,
        lr_schedule=lr_schedule, phased=phased,
        phase_length=phase_length, phase_epochs=phase_epochs,
        mask_full=mask_full, impute_mode=impute_mode, parallel_mse=parallel_mse,
        bayesian_head=bayesian_head,
    )
    solver = lambda c: solve_topk_min(c, k)

    cum_regret = 0
    regrets, flip_sizes = [], []
    contested_maes, noncontested_maes, decision_correct = [], [], []
    rank_corrs, contested_sigmas = [], []
    obs_counts = np.zeros(n_arms)
    selections_hist, optimals_hist = [], []

    for t in range(n_rounds):
        if context_dist == 'gaussian':
            x = rng.randn(input_dim)
        elif context_dist == 'uniform_narrow':
            x = rng.uniform(-0.5, 0.5, input_dim)
        elif context_dist == 'uniform_wide':
            x = rng.uniform(-3, 3, input_dim)
        else:
            x = rng.randn(input_dim)
        if cost_fn == 'linear':
            true_costs = (W_true @ x).flatten()
        else:
            true_costs = (1 / (1 + np.exp(-W_true @ x))).flatten()
        if ignore_context:
            x = np.zeros(input_dim)  # agent sees no context
        
        selection = agent.act(x)
        optimal = solver(true_costs)
        regret = true_costs @ selection - true_costs @ optimal
        cum_regret += max(regret, 0)
        regrets.append(cum_regret)
        
        obs_counts += (selection > 0.5)
        selections_hist.append((selection > 0.5).astype(int))
        optimals_hist.append((optimal > 0.5).astype(int))
        
        h = agent._features(x)
        pred_costs = agent._predict(h)
        stds = agent._uncertainty(h)
        
        _, n_flips = compute_flip_set(pred_costs, true_costs, k)
        flip_sizes.append(n_flips)
        
        sorted_true = np.sort(true_costs)
        if k < n_arms:
            boundary = (sorted_true[k-1] + sorted_true[k]) / 2
            margin = np.abs(true_costs - boundary)
            n_contested = min(2 * k, n_arms)
            contested_mask = np.zeros(n_arms, dtype=bool)
            contested_mask[np.argsort(margin)[:n_contested]] = True
        else:
            contested_mask = np.ones(n_arms, dtype=bool)
        
        errors = np.abs(pred_costs - true_costs)
        contested_maes.append(errors[contested_mask].mean())
        noncontested_mask = ~contested_mask
        if noncontested_mask.any():
            noncontested_maes.append(errors[noncontested_mask].mean())
        else:
            noncontested_maes.append(0.0)
        
        decision_correct.append(int(np.array_equal(
            (solver(pred_costs) > 0.5).astype(int),
            (optimal > 0.5).astype(int)
        )))
        
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            rho, _ = spearmanr(pred_costs, true_costs)
        rank_corrs.append(rho if np.isfinite(rho) else 0.0)
        contested_sigmas.append(stds[contested_mask].mean())
        
        agent.update(x, selection, true_costs, full_feedback=full_feedback)
    
    return {
        'regrets': np.array(regrets),
        'flip_sizes': np.array(flip_sizes),
        'contested_mae': np.array(contested_maes),
        'noncontested_mae': np.array(noncontested_maes),
        'decision_correct': np.array(decision_correct),
        'rank_corr': np.array(rank_corrs),
        'contested_sigma': np.array(contested_sigmas),
        'obs_counts': obs_counts,
        'selections': np.array(selections_hist),
        'optimals': np.array(optimals_hist),
    }

N_ARMS = 8
CONTEXT_DIM = 5
K = 2  # k/n = 25%
HIDDEN_DIM = 8

rng_gt = np.random.RandomState(42)
W_TRUE = rng_gt.randn(1, CONTEXT_DIM) + rng_gt.randn(N_ARMS, CONTEXT_DIM) * 0.4

def run_context_blind(W_true, k, n_rounds=3000, n_seeds=10, hidden_dim=8, lr=None):
    lr = lr or LR
    results = []
    for seed in range(n_seeds):
        r = run_experiment(W_true, k, n_rounds=n_rounds, hidden_dim=hidden_dim,
                           lr=lr, loss_type='mse', explore='greedy',
                           mask_grad=True, full_feedback=False, seed=seed,
                           ignore_context=True)
        results.append(r)
    return results

In [2]:
N_SEEDS = 10


def train_agent_for_viz(W_true, k, n_rounds=20, hidden_dim=4, loss_type='mse',
                        explore='greedy', mask_grad=True, full_feedback=False,
                        seed=0, beta=1.0, lr=None, lr_schedule='constant',
                        phased=False, phase_length=50, phase_epochs=5,
                        mask_full=False, impute_mode='model', parallel_mse=False,
                        bayesian_head=True, cost_fn='sigmoid'):
    """Train an agent and return it (for visualization of learned predictions)."""
    lr = lr or LR
    rng = np.random.RandomState(seed)
    n_arms, input_dim = W_true.shape

    agent = BayesianSharedFeatureAgent(
        input_dim, hidden_dim, n_arms, k, lr=lr,
        loss_type=loss_type, explore=explore, beta=beta,
        mask_grad=mask_grad, seed=seed, lr_schedule=lr_schedule,
        phased=phased, phase_length=phase_length, phase_epochs=phase_epochs,
        mask_full=mask_full, impute_mode=impute_mode, parallel_mse=parallel_mse,
        bayesian_head=bayesian_head,
    )

    for t in range(n_rounds):
        x = rng.randn(input_dim)
        if cost_fn == 'linear':
            true_costs = (W_true @ x).flatten()
        else:
            true_costs = (1 / (1 + np.exp(-W_true @ x))).flatten()
        selection = agent.act(x)
        agent.update(x, selection, true_costs, full_feedback=full_feedback)

    return agent


def plot_prediction_scatter(agent, W_true, title='', n_samples=300, seed=99,
                            show_context=False, cost_fn='sigmoid'):
    """Plot agent predictions."""
    rng = np.random.RandomState(seed)
    n_arms, input_dim = W_true.shape

    if show_context and input_dim == 1:
        x_samples = rng.randn(n_samples, 1)
        x_all, pred_all, arm_labels = [], [], []
        for xi in x_samples:
            h = agent._features(xi.flatten())
            pred_costs = agent._predict(h)
            for i in range(n_arms):
                x_all.append(xi[0])
                pred_all.append(pred_costs[i])
                arm_labels.append(f'Arm {i}')

        df = pd.DataFrame({'x': x_all, 'Predicted': pred_all, 'Arm': arm_labels})
        return (ggplot(df)
            + geom_point(aes('x', 'Predicted', color='Arm'), alpha=0.4, size=1.5)
            + geom_hline(yintercept=0.5, linetype='dashed', alpha=0.3)
            + coord_cartesian(xlim=(-3, 3), ylim=(-0.5, 1))
            + labs(x='Context x', y='Predicted cost', title=title)
            + theme(legend_position='none')
            + ggsize(400, 350)
        )
    else:
        preds_all, trues_all, arms_all = [], [], []
        for _ in range(n_samples):
            x = rng.randn(input_dim)
            if cost_fn == 'linear':
                true_costs = (W_true @ x).flatten()
            else:
                true_costs = (1 / (1 + np.exp(-W_true @ x))).flatten()
            h = agent._features(x)
            pred_costs = agent._predict(h)
            for i in range(n_arms):
                preds_all.append(pred_costs[i])
                trues_all.append(true_costs[i])
                arms_all.append(f'Arm {i}')

        df = pd.DataFrame({'True cost': trues_all, 'Predicted': preds_all, 'Arm': arms_all})
        lims = [min(min(trues_all), min(preds_all)) - 0.05,
                max(max(trues_all), max(preds_all)) + 0.05]
        return (ggplot(df)
            + geom_point(aes('True cost', 'Predicted', color='Arm'), alpha=0.3, size=1)
            + geom_abline(slope=1, intercept=0, linetype='dashed', color='black', alpha=0.5)
            + coord_fixed(xlim=lims, ylim=lims)
            + labs(title=title)
            + theme(legend_position='none')
            + ggsize(400, 350)
        )


def plot_decision_heatmap(result, title='', window=50):
    """Heatmap of per-arm selection frequency over time (rolling window)."""
    selections = result['selections']
    n_rounds, n_arms = selections.shape
    rows = []
    for t in range(window, n_rounds, max(1, n_rounds // 200)):
        freq = selections[t - window:t].mean(axis=0)
        for arm in range(n_arms):
            rows.append({'Round': t, 'Arm': f'Arm {arm}', 'Freq': freq[arm]})
    df = pd.DataFrame(rows)
    return (ggplot(df, aes('Round', 'Arm', fill='Freq'))
        + geom_tile()
        + scale_fill_gradient(low='white', high='#2C3E50', limits=[0, 1])
        + labs(title=title, fill='Select freq')
        + theme(legend_position='right')
        + ggsize(400, 250)
    )

In [3]:
try:
    display
except NameError:
    display = print  # fallback for non-notebook

def run_imputation_comparison(W_true, k, n_rounds=2000, hidden_dim=4,
                              configs=None, cost_fn='sigmoid', n_seeds=10):
    """Run all imputation configs and return plots + summary."""
    n_arms = W_true.shape[0]
    
    # Data distribution plot
    rng_data = np.random.RandomState(42)
    n_samples = 300
    if W_true.shape[1] == 1:
        x_samples = rng_data.randn(n_samples, 1)
        if cost_fn == 'linear':
            costs_samples = W_true @ x_samples.T
        else:
            costs_samples = 1 / (1 + np.exp(-W_true @ x_samples.T))
        rows = []
        for i in range(n_arms):
            for j in range(n_samples):
                rows.append({'x': x_samples[j, 0], 'cost': costs_samples[i, j], 'arm': f'Arm {i}'})
        df_data = pd.DataFrame(rows)
        p_data = (ggplot(df_data)
            + geom_point(aes('x', 'cost', color='arm'), alpha=0.4, size=1.5)
            + geom_hline(yintercept=0.0, linetype='dashed', alpha=0.3)
            + labs(x='Context x', y='True cost c(x)', title=f'Data ({cost_fn}, {n_arms} arms, k={k})', color='')
            + theme(legend_position='top')
            + ggsize(400, 350)
        )
    else:
        p_data = None
    
    if configs is None:
        configs = [
            ('BayesianSPO (full)',              dict(loss_type='spo', full_feedback=True)),
            ('BayesianSPO (partial)',           dict(loss_type='spo', full_feedback=False)),
            ('BayesianSPO (partial, mean imp)', dict(loss_type='spo', full_feedback=False, impute_mode='mean')),
            ('BayesianSPO (post M=1)',          dict(loss_type='spo', full_feedback=False, impute_mode='posterior')),
            ('SPO (emp UCB)',                   dict(loss_type='spo', full_feedback=False, explore='emp_ucb')),
            ('SPO (parallel MSE impute)',       dict(loss_type='spo', full_feedback=False,
                                                     impute_mode='mse_parallel', parallel_mse=True)),
            ('MSE (partial)',                   dict(loss_type='mse', full_feedback=False)),
        ]
    
    show_ctx = (W_true.shape[1] == 1)
    scatter_plots = [p_data] if p_data is not None else []
    regret_rows = []
    
    for name, kw in configs:
        kw2 = dict(kw)
        explore_kw = kw2.pop('explore', 'ucb')
        agent = train_agent_for_viz(W_true, k, n_rounds=n_rounds,
                                    hidden_dim=hidden_dim, explore=explore_kw, mask_grad=True,
                                    seed=0, cost_fn=cost_fn, **kw2)
        scatter_plots.append(
            plot_prediction_scatter(agent, W_true, title=name, n_samples=300,
                                    seed=99, show_context=show_ctx, cost_fn=cost_fn)
        )
        for seed in range(n_seeds):
            r = run_experiment(W_true, k, n_rounds=n_rounds, hidden_dim=hidden_dim,
                               lr=LR, explore=explore_kw, mask_grad=True, seed=seed,
                               cost_fn=cost_fn, **kw2)
            cum_regret = r['regrets']
            inst_regret = np.diff(cum_regret, prepend=0)
            for t in range(n_rounds):
                regret_rows.append({'Round': t, 'Cumulative Regret': cum_regret[t],
                                    'Instant Regret': inst_regret[t], 'Agent': name, 'Seed': seed})
    
    df_regret = pd.DataFrame(regret_rows)
    agg = df_regret.groupby(['Round', 'Agent']).agg(
        cum_mean=('Cumulative Regret', 'mean'),
        inst_mean=('Instant Regret', 'mean'),
    ).reset_index()
    
    # Display scatter plots
    for i in range(0, len(scatter_plots), 5):
        row = gggrid(scatter_plots[i:i+5], ncol=5) + ggsize(1800, 380)
        display(row)
    
    # Regret curves
    p_cum = (ggplot(agg, aes('Round', 'cum_mean', color='Agent'))
        + geom_line(size=1.2)
        + labs(x='Round', y='Cumulative Regret', title='Cumulative Regret')
        + theme(legend_position='none')
        + ggsize(800, 350))
    p_inst = (ggplot(agg, aes('Round', 'inst_mean', color='Agent'))
        + geom_line(size=1.2, alpha=0.8)
        + labs(x='Round', y='Per-Round Regret', title='Per-Round Regret')
        + theme(legend_position='bottom')
        + ggsize(800, 350))
    display(gggrid([p_cum, p_inst], ncol=2) + ggsize(1800, 380))
    
    # Summary table
    for name, kw in configs:
        final = df_regret[(df_regret['Agent'] == name) & (df_regret['Round'] == n_rounds - 1)]
        regs = final['Cumulative Regret'].values
        print(f"{name:30s}: regret = {np.mean(regs):.1f} +/- {np.std(regs):.1f}")
    
    return df_regret


## 1. Well-Specified Setting: Linear Costs

Costs are  — the linear feature model can represent this perfectly.
Any failure here is purely due to partial feedback + imputation, not model misspecification.

In [4]:
W_2ARM = np.array([[1.0], [2.5]])

configs_main = [
    # BayesianSPO variants (Bayesian posterior head for prediction + uncertainty)
    ('BayesSPO (full)',                 dict(loss_type='spo', full_feedback=True)),
    ('BayesSPO (full, force mask)',     dict(loss_type='spo', full_feedback=True, mask_full=True)),
    ('BayesSPO (partial)',              dict(loss_type='spo', full_feedback=False)),
    ('BayesSPO (partial, mean imp)',    dict(loss_type='spo', full_feedback=False, impute_mode='mean')),
    ('BayesSPO (post M=1)',             dict(loss_type='spo', full_feedback=False, impute_mode='posterior')),
    ('BayesSPO-Phased (partial)',       dict(loss_type='spo', full_feedback=False, phased=True)),
    # BayesSPO with empirical count-based UCB (Bayesian prediction head still)
    ('BayesSPO (emp UCB)',              dict(loss_type='spo', full_feedback=False, explore='emp_ucb')),
    # Pure SPO with point-estimate head + empirical UCB (no Bayesian layer anywhere)
    ('SPO (emp UCB)',                   dict(loss_type='spo', full_feedback=False,
                                             explore='emp_ucb', bayesian_head=False)),
    # SPO for decisions + parallel MSE (per-arm OLS on raw x) for imputation
    ('SPO (parallel MSE impute)',       dict(loss_type='spo', full_feedback=False,
                                             impute_mode='mse_parallel', parallel_mse=True)),
    # Baselines
    ('MSE (full)',                      dict(loss_type='mse', full_feedback=True)),
    ('MSE (partial)',                   dict(loss_type='mse', full_feedback=False)),
    ('MSE-Phased (partial)',            dict(loss_type='mse', full_feedback=False, phased=True)),
    ('Hybrid (MSE online, SPO+ offline)', dict(loss_type='hybrid_phased', full_feedback=False, phased=True)),
]

df_linear = run_imputation_comparison(W_2ARM, k=1, n_rounds=2000, hidden_dim=4,
                                       configs=configs_main, cost_fn='linear')

BayesSPO (full)               : regret = 0.4 +/- 0.4
BayesSPO (full, force mask)   : regret = 0.4 +/- 0.4
BayesSPO (partial)            : regret = 8.2 +/- 6.0
BayesSPO (partial, mean imp)  : regret = 137.7 +/- 29.8
BayesSPO (post M=1)           : regret = 8.1 +/- 6.0
BayesSPO-Phased (partial)     : regret = 7.8 +/- 6.0
BayesSPO (emp UCB)            : regret = 5.8 +/- 5.4
SPO (emp UCB)                 : regret = 26.7 +/- 7.3
SPO (parallel MSE impute)     : regret = 339.6 +/- 176.7
MSE (full)                    : regret = 0.2 +/- 0.2
MSE (partial)                 : regret = 8.3 +/- 5.9
MSE-Phased (partial)          : regret = 7.8 +/- 6.0
Hybrid (MSE online, SPO+ offline): regret = 8.1 +/- 5.9


## 2. Misspecified Setting: Sigmoid Costs

Costs are  — nonlinear, linear features cannot represent this.
Combines partial feedback imputation problems with model misspecification.

In [5]:
W_2ARM = np.array([[1.0], [2.5]])

df_sigmoid = run_imputation_comparison(W_2ARM, k=1, n_rounds=2000, hidden_dim=4,
                                        configs=configs_main, cost_fn='sigmoid')

BayesSPO (full)               : regret = 0.1 +/- 0.0
BayesSPO (full, force mask)   : regret = 0.1 +/- 0.0
BayesSPO (partial)            : regret = 135.2 +/- 2.6
BayesSPO (partial, mean imp)  : regret = 113.6 +/- 2.4
BayesSPO (post M=1)           : regret = 133.6 +/- 2.4
BayesSPO-Phased (partial)     : regret = 81.0 +/- 30.0
BayesSPO (emp UCB)            : regret = 33.7 +/- 6.5
SPO (emp UCB)                 : regret = 105.3 +/- 9.2
SPO (parallel MSE impute)     : regret = 119.4 +/- 4.6
MSE (full)                    : regret = 123.3 +/- 2.6
MSE (partial)                 : regret = 129.8 +/- 2.8
MSE-Phased (partial)          : regret = 99.1 +/- 12.3
Hybrid (MSE online, SPO+ offline): regret = 67.6 +/- 13.3


## 3. Scaling: More Arms

Does posterior imputation help with more arms (more observed data per round)?

In [6]:
rng_gt = np.random.RandomState(42)

configs_scale = [
    ('BayesSPO (full)',             dict(loss_type='spo', full_feedback=True)),
    ('BayesSPO (partial)',          dict(loss_type='spo', full_feedback=False)),
    ('BayesSPO-Phased',             dict(loss_type='spo', full_feedback=False, phased=True)),
    ('BayesSPO (emp UCB)',          dict(loss_type='spo', full_feedback=False, explore='emp_ucb')),
    ('SPO (emp UCB)',               dict(loss_type='spo', full_feedback=False,
                                         explore='emp_ucb', bayesian_head=False)),
    ('SPO (parallel MSE impute)',   dict(loss_type='spo', full_feedback=False,
                                         impute_mode='mse_parallel', parallel_mse=True)),
    ('MSE (full)',                  dict(loss_type='mse', full_feedback=True)),
    ('MSE (partial)',               dict(loss_type='mse', full_feedback=False)),
    ('MSE-Phased (partial)',        dict(loss_type='mse', full_feedback=False, phased=True)),
    ('Hybrid (MSE+SPO+)',           dict(loss_type='hybrid_phased', full_feedback=False, phased=True)),
]

#                  (n_arms, k, hidden_dim, n_rounds, W_spread, label)
scale_cases = [
    (4,  2, 8,  2000, 0.4, '4 arms, k=2'),
    (8,  4, 8,  2000, 0.4, '8 arms, k=4'),
    (16, 4, 16, 5000, 0.4, '16 arms, k=4'),
]

for n_arms, k, hd, n_rounds, spread, label in scale_cases:
    W = rng_gt.randn(1, 5) + rng_gt.randn(n_arms, 5) * spread
    print('{} (sigmoid, hd={}, T={})'.format(label, hd, n_rounds))
    for name, kw in configs_scale:
        regs = []
        for seed in range(5):
            # Pull explore out of kw if present, else default ucb
            kw2 = dict(kw)
            explore = kw2.pop('explore', 'ucb')
            r = run_experiment(W, k, n_rounds=n_rounds, hidden_dim=hd, lr=LR,
                               explore=explore, mask_grad=True, seed=seed,
                               cost_fn='sigmoid', **kw2)
            regs.append(r['regrets'][-1])
        print('  {:30s}: regret = {:.1f} +/- {:.1f}'.format(name, np.mean(regs), np.std(regs)))
    print()

4 arms, k=2 (sigmoid, hd=8, T=2000)
  BayesSPO (full)               : regret = 1.6 +/- 0.3
  BayesSPO (partial)            : regret = 24.1 +/- 3.0
  BayesSPO-Phased               : regret = 10.8 +/- 0.6
  BayesSPO (emp UCB)            : regret = 21.7 +/- 2.3
  SPO (emp UCB)                 : regret = 152.2 +/- 14.6
  SPO (parallel MSE impute)     : regret = 244.9 +/- 5.7
  MSE (full)                    : regret = 7.2 +/- 3.2
  MSE (partial)                 : regret = 42.0 +/- 7.3
  MSE-Phased (partial)          : regret = 13.1 +/- 1.2
  Hybrid (MSE+SPO+)             : regret = 10.9 +/- 0.8

8 arms, k=4 (sigmoid, hd=8, T=2000)
  BayesSPO (full)               : regret = 4.9 +/- 1.1
  BayesSPO (partial)            : regret = 66.4 +/- 6.2
  BayesSPO-Phased               : regret = 29.0 +/- 1.6
  BayesSPO (emp UCB)            : regret = 58.0 +/- 5.9
  SPO (emp UCB)                 : regret = 375.2 +/- 13.2
  SPO (parallel MSE impute)     : regret = 586.7 +/- 5.3
  MSE (full)                